# Learned Stage-Wise Curvature Tuning on Flowers102

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Oxford Flowers-102 dataset using an ImageNet-pretrained ResNet-18.

Flowers102 is a fine-grained image-classification dataset containing many visually similar flower categories. Testing Learned SW-CT on this dataset helps evaluate whether stage-wise curvature adaptation remains useful when the target task requires distinguishing between closely related visual classes.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves accuracy or changes the learned stage-wise beta values.

Wall-clock execution time is recorded for each method to compare computational cost.

In [1]:
!nvidia-smi

Thu Oct  1 02:26:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 7.10 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 50.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 59.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 66.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 M

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Flowers102 Dataset

Before running the experiments, the Flowers102 data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_flowers102",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

100%|██████████| 345M/345M [00:15<00:00, 21.8MB/s] 
100%|██████████| 502/502 [00:00<00:00, 1.65MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 39.5MB/s]

Train samples: 1020
Validation samples: 1020
Test samples: 6149


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([50, 62, 15, 96, 28,  7, 87, 73, 13, 84])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Flowers102 dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:28:39.666 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed42_epochs20.log
2026-10-01 02:28:39.669 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 148MB/s]
2026-10-01 02:28:40.803 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:28:40.821 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 02:28:40.822 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:28:57.992 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.757530, Accuracy: 3.12%
2026-10-01 02:28:58.298 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.228443, Val Accuracy: 9.02%
2026-10-01 02:28:58.299 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.0

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:41:01.820 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed43_epochs20.log
2026-10-01 02:41:01.823 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 02:41:02.590 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:41:02.614 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 02:41:02.615 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:41:14.472 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.895680, Accuracy: 0.00%
2026-10-01 02:41:14.791 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.170737, Val Accuracy: 9.22%
2026-10-01 02:41:14.792 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.22 at epoch 1
2026-10-01 02:41:14.866 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.046525, Accuracy: 21.88%
2026-10-01 02:41:15.169 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.259589, Val

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:54:27.829 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed44_epochs20.log
2026-10-01 02:54:27.833 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 02:54:28.740 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:54:28.766 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 02:54:28.766 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:54:43.241 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.680496, Accuracy: 0.00%
2026-10-01 02:54:43.570 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.151682, Val Accuracy: 12.65%
2026-10-01 02:54:43.571 | INFO     | train:linear_probe:190 - New best validation accuracy: 12.65 at epoch 1
2026-10-01 02:54:43.645 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.038638, Accuracy: 18.75%
2026-10-01 02:54:43.972 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.243027, V

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:07:59.744 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:07:59.874 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:08:00.656 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:08:00.657 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:08:00.705 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:08:00.706 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:08:00.706 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:08:02.504 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.904130, Accuracy: 0.00%
2026-10-01 03:08:16.070 | INFO     | train:test_epoch:90 

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:13:30.612 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:13:30.733 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:13:31.459 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:13:31.459 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:13:31.493 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:13:31.494 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:13:31.494 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:13:33.037 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.813028, Accuracy: 0.00%
2026-10-01 03:13:46.549 | INFO     | train:test_epoch:90 

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:18:56.858 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:18:56.949 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:18:57.628 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:18:57.628 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:18:57.664 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:18:57.665 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:18:57.665 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:18:58.951 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.124666, Accuracy: 0.00%
2026-10-01 03:19:10.645 | INFO     | train:test_epoch:90 

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:24:28.791 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-01 03:24:28.858 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:24:29.493 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:24:29.494 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:24:29.538 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 03:24:29.538 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:24:29.539 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:24:30.983 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.829558, Accuracy: 0.00%
2026-10-01 03:24:42.791 | INFO     | train:te

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:30:00.608 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-01 03:30:00.689 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:30:01.357 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:30:01.357 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:30:01.392 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 03:30:01.392 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:30:01.393 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:30:02.709 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.838178, Accuracy: 0.00%
2026-10-01 03:30:14.338 | INFO     | train:

In [11]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:35:31.604 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-01 03:35:31.704 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:35:32.386 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:35:32.386 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:35:32.432 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 03:35:32.432 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:35:32.433 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 03:35:33.785 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.071203, Accuracy: 0.00%
2026-10-01 03:35:45.585 | INFO     | train:

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 03:41:03.766 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed42_epochs30.log
2026-10-01 03:41:03.769 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 03:41:04.533 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 03:41:04.554 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 03:41:04.554 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 03:41:17.164 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.757530, Accuracy: 3.12%
2026-10-01 03:41:17.584 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.228443, Val Accuracy: 9.02%
2026-10-01 03:41:17.585 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.02 at epoch 1
2026-10-01 03:41:17.691 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.029564, Accuracy: 15.62%
2026-10-01 03:41:18.060 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.319124, Val

In [13]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 03:57:25.076 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed43_epochs30.log
2026-10-01 03:57:25.078 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 03:57:25.825 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 03:57:25.850 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 03:57:25.850 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 03:57:40.021 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.895680, Accuracy: 0.00%
2026-10-01 03:57:40.402 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.170737, Val Accuracy: 9.22%
2026-10-01 03:57:40.404 | INFO     | train:linear_probe:190 - New best validation accuracy: 9.22 at epoch 1
2026-10-01 03:57:40.511 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.046525, Accuracy: 21.88%
2026-10-01 03:57:40.915 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.259589, Val

In [14]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 04:13:37.991 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet18_seed44_epochs30.log
2026-10-01 04:13:37.994 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 04:13:38.889 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 04:13:38.921 | INFO     | __main__:main:116 - Number of trainable parameters: 52326
2026-10-01 04:13:38.921 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 04:13:53.539 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.680496, Accuracy: 0.00%
2026-10-01 04:13:53.868 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.151682, Val Accuracy: 12.65%
2026-10-01 04:13:53.869 | INFO     | train:linear_probe:190 - New best validation accuracy: 12.65 at epoch 1
2026-10-01 04:13:53.948 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.038638, Accuracy: 18.75%
2026-10-01 04:13:54.251 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 3.243027, V

In [15]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 04:29:59.406 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 04:29:59.496 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 04:30:00.162 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 04:30:00.162 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 04:30:00.193 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 04:30:00.194 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 04:30:00.194 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 04:30:01.518 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.904130, Accuracy: 0.00%
2026-10-01 04:30:12.919 | INFO     | train:test_epoch:90 

In [16]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 04:37:45.679 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 04:37:45.795 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 04:37:46.510 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 04:37:46.511 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 04:37:46.546 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 04:37:46.547 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 04:37:46.548 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 04:37:48.178 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.813028, Accuracy: 0.00%
2026-10-01 04:38:02.024 | INFO     | train:test_epoch:90 

In [17]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 04:45:35.429 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 04:45:35.537 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 04:45:36.276 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 04:45:36.277 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 04:45:36.331 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 04:45:36.332 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 04:45:36.332 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 04:45:37.944 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.124666, Accuracy: 0.00%
2026-10-01 04:45:52.239 | INFO     | train:test_epoch:90 

In [18]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 04:54:16.498 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 04:54:16.608 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 04:54:17.370 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 04:54:17.371 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 04:54:17.407 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 04:54:17.407 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 04:54:17.408 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 04:54:18.958 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.829558, Accuracy: 0.00%
2026-10-01 04:54:33.231 | INFO     | train:te

In [19]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 05:02:44.931 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-01 05:02:45.009 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 05:02:45.657 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 05:02:45.658 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 05:02:45.690 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 05:02:45.691 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 05:02:45.691 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 05:02:46.993 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.838178, Accuracy: 0.00%
2026-10-01 05:02:58.654 | INFO     | train:

In [20]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 05:11:10.043 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-01 05:11:10.180 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 05:11:10.745 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 05:11:10.745 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 05:11:10.779 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 05:11:10.780 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 05:11:10.780 | INFO     | __main__:main:439 - Total trainable parameters: 52330
2026-10-01 05:11:11.946 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.071203, Accuracy: 0.00%
2026-10-01 05:11:26.587 | INFO     | train: